In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../House_Rent_Dataset.csv")

In [3]:
df["Floor"] = df["Floor"].astype(str).str.split(" out of").str[0]
df["Floor"] = df["Floor"].replace({"Ground": 0, "Upper Basement": -1, "Lower Basement": -2})
df["Floor"] = pd.to_numeric(df["Floor"], errors="coerce").fillna(0)

In [4]:
df.shape


(4746, 12)

In [5]:

df["Rent"].describe()

count    4.746000e+03
mean     3.499345e+04
std      7.810641e+04
min      1.200000e+03
25%      1.000000e+04
50%      1.600000e+04
75%      3.300000e+04
max      3.500000e+06
Name: Rent, dtype: float64

In [6]:
"Most expensive listings:"



'Most expensive listings:'

In [7]:
df.sort_values("Rent", ascending=False).head(10)[["BHK", "Size", "City", "Furnishing Status", "Rent"]]

,BHK,Size,City,Furnishing Status,Rent
1837,3,2500,Bangalore,Semi-Furnished,3500000
1001,4,5000,Mumbai,Semi-Furnished,1200000
827,4,3064,Mumbai,Semi-Furnished,1000000
1329,4,3200,Mumbai,Furnished,850000
1459,4,3200,Mumbai,Furnished,700000
1484,4,1962,Mumbai,Semi-Furnished,680000
1319,5,3000,Mumbai,Semi-Furnished,650000
1384,5,4500,Mumbai,Furnished,600000
726,4,2500,Mumbai,Semi-Furnished,600000
3656,2,950,Chennai,Unfurnished,600000


In [8]:
"Cheapest listings:"

'Cheapest listings:'

In [9]:
df.sort_values("Rent").head(10)[["BHK", "Size", "City", "Furnishing Status", "Rent"]]

,BHK,Size,City,Furnishing Status,Rent
4076,3,2100,Hyderabad,Furnished,1200
285,1,200,Kolkata,Semi-Furnished,1500
471,1,500,Kolkata,Semi-Furnished,1800
2475,2,60,Delhi,Unfurnished,2000
146,2,550,Kolkata,Furnished,2200
506,1,700,Kolkata,Unfurnished,2200
4220,1,80,Hyderabad,Semi-Furnished,3000
3068,1,350,Chennai,Unfurnished,3000
306,1,500,Kolkata,Semi-Furnished,3000
3378,1,350,Chennai,Unfurnished,3000


In [ ]:
df["rent_per_sqft"] = df["Rent"] / df["Size"]
df["sqft_per_bhk"] = df["Size"] / df["BHK"]
df[["rent_per_sqft", "sqft_per_bhk"]].describe()

,rent_per_sqft,sqft_per_bhk
count,4746.000000,4746.000000
mean,39.257087,458.454263
std,59.855230,211.596397
min,0.571429,3.333333
25%,13.333333,350.000000
50%,20.000000,450.000000
75%,42.005986,550.000000
max,1500.000000,8000.000000


In [ ]:
city_median = df.groupby("City")["rent_per_sqft"].transform("median")
df["rent_vs_city"] = df["rent_per_sqft"] / city_median
df.groupby("City")["rent_per_sqft"].median()


City
Bangalore    17.500000
Chennai      16.216216
Delhi        28.571429
Hyderabad    14.545455
Kolkata      13.315603
Mumbai       71.428571
Name: rent_per_sqft, dtype: float64

In [12]:
df["rent_vs_city"].describe()

count    4746.000000
mean        1.430316
std         2.800201
min         0.039286
25%         0.756250
50%         1.000000
75%         1.400000
max       103.125000
Name: rent_vs_city, dtype: float64

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
df["log_rent_vs_city"] = np.log(df["rent_vs_city"])
df["log_sqft_per_bhk"] = np.log(df["sqft_per_bhk"])
features = ["log_rent_vs_city", "log_sqft_per_bhk"]
iso = IsolationForest(contamination=0.03, random_state=42)
df["unusual"] = iso.fit_predict(df[features])
df["score"] = iso.decision_function(df[features])
print(df["unusual"].value_counts())
print(df.loc[[4076, 2475, 1837, 3622], ["score", "unusual"]])

unusual
 1    4603
-1     143
Name: count, dtype: int64
         score  unusual
4076 -0.001607       -1
2475 -0.028834       -1
1837 -0.064962       -1
3622  0.071662        1


In [ ]:
df["score"] = iso.decision_function(df[features])   # lower = more unusual
print("Cutoff (highest score among flagged):", df.loc[df["unusual"] == -1, "score"].max())
print(df.loc[[4076, 2475, 1837, 3622], ["score", "unusual"]])
print("Rows with rent < 20% of city rate:", (df["rent_vs_city"] < 0.2).sum())

Cutoff (highest score among flagged): -1.8589730019114015e-05
         score  unusual
4076 -0.001607       -1
2475 -0.028834       -1
1837 -0.064962       -1
3622  0.071662        1
Rows with rent < 20% of city rate: 10


In [ ]:
flagged = df[df["unusual"] == -1]
flagged.sort_values("rent_vs_city", ascending=False) [["BHK", "Size", "City", "Rent", "rent_vs_city", "sqft_per_bhk"]].head(15)

,BHK,Size,City,Rent,rent_vs_city,sqft_per_bhk
4653,3,10,Hyderabad,15000,103.125000,3.333333
1837,3,2500,Bangalore,3500000,80.000000,833.333333
4004,2,130,Hyderabad,130000,68.750000,65.000000
3922,3,25,Hyderabad,23000,63.250000,8.333333
3656,2,950,Chennai,600000,38.947368,475.000000
2656,5,200,Delhi,190000,33.250000,40.000000
462,3,120,Kolkata,34000,21.278296,40.000000
116,1,20,Kolkata,5000,18.774967,20.000000
2904,3,300,Delhi,150000,17.500000,100.000000
4117,3,150,Hyderabad,37000,16.958333,50.000000


In [16]:
flagged.sort_values("rent_vs_city")[["BHK", "Size", "City", "Rent", "rent_vs_city", "sqft_per_bhk"]].head(15)

,BHK,Size,City,Rent,rent_vs_city,sqft_per_bhk
4076,3,2100,Hyderabad,1200,0.039286,700.000000
267,1,2160,Kolkata,6000,0.208611,2160.000000
238,1,1800,Kolkata,5500,0.229472,1800.000000
231,1,1000,Kolkata,3500,0.262850,1000.000000
2109,1,1200,Bangalore,6000,0.285714,1200.000000
3091,2,2800,Chennai,15000,0.330357,1400.000000
3601,1,1900,Chennai,15000,0.486842,1900.000000
2475,2,60,Delhi,2000,1.166667,30.000000
2624,2,130,Delhi,8000,2.153846,65.000000
2454,2,145,Delhi,9000,2.172414,72.500000


In [17]:
check = [4076, 2475, 1837, 3622]
df.loc[check, ["BHK", "Size", "City", "Rent", "rent_vs_city", "sqft_per_bhk", "unusual"]]

,BHK,Size,City,Rent,rent_vs_city,sqft_per_bhk,unusual
4076,3,2100,Hyderabad,1200,0.039286,700.000000,-1
2475,2,60,Delhi,2000,1.166667,30.000000,-1
1837,3,2500,Bangalore,3500000,80.000000,833.333333,-1
3622,5,6000,Chennai,100000,1.027778,1200.000000,1


In [18]:
borderline = df[df["unusual"] == -1].sort_values("score", ascending=False).head(15)
print(borderline[["BHK", "Size", "City", "Furnishing Status", "Rent", "rent_vs_city", "sqft_per_bhk", "score"]])

      BHK  Size       City Furnishing Status    Rent  rent_vs_city  \
3924    2   200  Hyderabad       Unfurnished    6500      2.234375   
4436    2    90  Hyderabad       Unfurnished   10000      7.638889   
4238    2   150  Hyderabad       Unfurnished    8000      3.666667   
4421    2   160  Hyderabad       Unfurnished    9000      3.867188   
2772    3   250      Delhi    Semi-Furnished   30000      4.200000   
2685    2    80      Delhi    Semi-Furnished   17000      7.437500   
4293    2   167  Hyderabad       Unfurnished    9500      3.910928   
4076    3  2100  Hyderabad         Furnished    1200      0.039286   
2990    3  3000    Chennai         Furnished  200000      4.111111   
2717    2    90      Delhi    Semi-Furnished   16500      6.416667   
2857    3  3200      Delhi    Semi-Furnished  350000      3.828125   
2657    2    95      Delhi    Semi-Furnished   13000      4.789474   
2835    3   120      Delhi         Furnished   18000      5.250000   
2497    2   120     

In [ ]:
import joblib
city_medians = df.groupby("City")["rent_per_sqft"].median().to_dict()
joblib.dump({"model": iso, "city_medians": city_medians}, "unusual_model.pkl")
print(city_medians)

{'Bangalore': 17.5, 'Chennai': 16.216216216216218, 'Delhi': 28.571428571428573, 'Hyderabad': 14.545454545454545, 'Kolkata': 13.315602836879432, 'Mumbai': 71.42857142857143}


In [ ]:
from predict import check_listing
print(check_listing({"BHK": 3, "Size": 2100, "City": "Hyderabad", "Rent": 1200}))   # row 4076
print(check_listing({"BHK": 2, "Size": 900,  "City": "Mumbai",    "Rent": 65000}))  # normal
print(check_listing({"BHK": 2, "Size": 900,  "City": "Pune",      "Rent": 20000}))  # unknown city

{'unusual': True}
{'unusual': False}
{'unusual': None, 'message': 'City not supported'}
